Importing the libaries


In [ ]:
# Importing the necessary libraries

# Enable automatic reloading of modules when they are updated
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
import textwrap

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import Image, display
from torchmetrics.text.bleu import BLEUScore

from src.train import (
    build_sequence_dataloaders,
    build_tokenizer,
    load_storyreasoning,
    train_experiment3,

   
)
from src.utils import (
    ensure_dirs,
    generate,
    load_config,
    set_seed,
    validation,
)

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(str(CONFIG_PATH))

CONFIG_PATH = "config.yaml"
config = load_config(CONFIG_PATH)
set_seed(config.get("seed", 42))
ensure_dirs(config["paths"]["checkpoint_dir"], config["paths"]["results_dir"])
output_dir = Path(config["paths"]["results_dir"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


Loading and Saving Data

In [ ]:
# Loading the dataset
tokenizer = build_tokenizer()
train_dataset, test_dataset = load_storyreasoning(config)
train_dataloader, val_dataloader, test_dataloader = build_sequence_dataloaders(
    config,
    tokenizer,
    train_dataset,
    test_dataset,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")


In [ ]:
"""
A sanity check cell to verify the data pipeline.
It grabs a single batch from the training dataset and prints the shapes of the returned tensors (images, descriptions, etc.) to ensure everything is loaded correctly.
"""

frames, descriptions, image_target, text_target, roi1, roi2, roi_valid, roi_frame, ent_id = next(iter(train_dataloader))

print("frames:", frames.shape)
print("descriptions:", descriptions.shape)
print("image_target:", image_target.shape)
print("text_target:", text_target.shape)
print("roi_valid:", roi_valid.shape)


Experiment 3 - Sequence Predictor: Bidirectional LSTM


In [ ]:
experiment_name = "Experiment_3"
experiment3_dir = Path("results") / experiment_name
output_dir = experiment3_dir
ensure_dirs(experiment3_dir)
print(f"Experiment 3 outputs: {experiment3_dir}")

Experiment 3 Training


In [ ]:
sequence_predictor, tokenizer, val_dataloader, losses, training_log = train_experiment3(
    CONFIG_PATH,
    show_validation=True,
)


Experiment 3 - Saving the Training Logs

In [ ]:

experiment3_dir = Path("results") / "Experiment_3"
output_dir = experiment3_dir
ensure_dirs(experiment3_dir)
log_path = experiment3_dir / "training_log.txt"

with open(log_path, "w", encoding="utf-8") as f:
    for line in training_log:
        print(line)
        f.write(line + "\n")

print(f"Training log saved: {log_path}")


Validation Run

In [ ]:

validation(
    sequence_predictor,
    val_dataloader,
    tokenizer,
    device,
    show=True,
)
sequence_predictor.eval()

Generated Text Example

In [ ]:
# @title Generated Text Example for Experiment 3
# This cell prints a qualitative BiLSTM prediction example without saving extra raw images.
experiment3_dir = Path("results") / "Experiment_3"
output_dir = experiment3_dir
ensure_dirs(experiment3_dir)
prediction_text_path = experiment3_dir / "prediction_text_example.txt"

sequence_predictor.eval()
frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))
frames = frames.to(device)
descriptions = descriptions.to(device)
text_target = text_target.to(device)

with torch.no_grad():
    _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)
    generated_tokens = generate(
        sequence_predictor.text_decoder,
        h0[:, 0, :].unsqueeze(1),
        c0[:, 0, :].unsqueeze(1),
        max_len=150,
        sos_token_id=tokenizer.cls_token_id,
        eos_token_id=tokenizer.sep_token_id,
        device=device,
    )

text_target_decode = text_target.squeeze(1) if text_target.dim() == 3 else text_target
true_sentence = tokenizer.decode(text_target_decode[0].cpu(), skip_special_tokens=True)
pred_sentence = tokenizer.decode(generated_tokens, skip_special_tokens=True)

with open(prediction_text_path, "w", encoding="utf-8") as f:
    f.write("Experiment 3 Prediction Example\n")
    f.write("=" * 40 + "\n")
    f.write(f"Target Text:\n{true_sentence}\n\n")
    f.write(f"Experiment 3 Predicted Text:\n{pred_sentence}\n")

print("Target Text:", true_sentence)
print("Experiment 3 Predicted Text:", pred_sentence)
print(f"Prediction text example saved: {prediction_text_path}")


Loss Curve

In [ ]:

plot_path = experiment3_dir / "losscurve.png"

plt.figure(figsize=(8, 5))
plt.plot(losses, label="Experiment 3 Training Loss", color="blue", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experiment 3: Loss Curve")
plt.legend()
plt.grid(True)

plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print(f"Loss Curve Saved: {plot_path}")


Generate Predictions

In [ ]:
experiment3_dir = Path("results") / "Experiment_3"
output_dir = experiment3_dir
ensure_dirs(experiment3_dir)

sequence_predictor.eval()
pred_sentences = []
true_sentences = []
max_batches = config.get("evaluation", {}).get("bleu_max_batches", 5)

with torch.no_grad():
    for batch_index, (frames, descriptions, image_target, text_target, *_ ) in enumerate(val_dataloader):
        if max_batches is not None and batch_index >= max_batches:
            break

        frames = frames.to(device)
        descriptions = descriptions.to(device)
        text_target = text_target.to(device)

        _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)

        for i in range(frames.size(0)):
            generated_tokens = generate(
                sequence_predictor.text_decoder,
                h0[:, i, :].unsqueeze(1),
                c0[:, i, :].unsqueeze(1),
                max_len=150,
                sos_token_id=tokenizer.cls_token_id,
                eos_token_id=tokenizer.sep_token_id,
                device=device,
            )
            pred_sentences.append(tokenizer.decode(generated_tokens, skip_special_tokens=True))

        if text_target.dim() == 3:
            text_target_decode = text_target.squeeze(1)
        else:
            text_target_decode = text_target

        for seq in text_target_decode:
            true_sentences.append(tokenizer.decode(seq.cpu().numpy(), skip_special_tokens=True))

print("Number of predictions:", len(pred_sentences))
print("Number of true sentences:", len(true_sentences))


Metrics Calculation & Table Saving

In [ ]:
reference = [[s] for s in true_sentences]
bleu1_metric = BLEUScore(n_gram=1)
bleu4_metric = BLEUScore(n_gram=4)
experiment3_bleu_val = bleu1_metric(pred_sentences, reference).item()
experiment3_bleu4_val = bleu4_metric(pred_sentences, reference).item()


def simple_meteor_score(prediction, reference_sentence):
    pred_tokens = prediction.lower().split()
    ref_tokens = reference_sentence.lower().split()
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}
    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1
    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    matches = sum(min(pred_counts.get(token, 0), ref_counts.get(token, 0)) for token in pred_counts)
    if matches == 0:
        return 0.0

    precision = matches / len(pred_tokens)
    recall = matches / len(ref_tokens)
    return (10 * precision * recall) / (recall + 9 * precision + 1e-8)


try:
    from nltk.translate.meteor_score import meteor_score

    meteor_values = [
        meteor_score([true.lower().split()], pred.lower().split())
        for pred, true in zip(pred_sentences, true_sentences)
    ]
except Exception:
    meteor_values = [
        simple_meteor_score(pred, true)
        for pred, true in zip(pred_sentences, true_sentences)
    ]

experiment3_meteor_val = sum(meteor_values) / len(meteor_values) if meteor_values else 0.0

print("\n" + "=" * 50)
print("Experiment 3 Results")
print("=" * 50)
print(f"Final Training Loss: {losses[-1]:.4f}")
print(f"BLEU Score: {experiment3_bleu_val:.4f}")
print(f"BLEU-4 Score: {experiment3_bleu4_val:.4f}")
print(f"METEOR Score: {experiment3_meteor_val:.4f}")
print("=" * 50 + "\n")

metrics_path = experiment3_dir / "metrics.txt"
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("Experiment 3\n")
    f.write("=" * 40 + "\n")
    f.write(f"{'Metric':<25} | {'Value':<10}\n")
    f.write("-" * 40 + "\n")
    f.write(f"{'Final Training Loss':<25} | {losses[-1]:.4f}\n")
    f.write(f"{'BLEU Score':<25} | {experiment3_bleu_val:.4f}\n")
    f.write(f"{'BLEU-4 Score':<25} | {experiment3_bleu4_val:.4f}\n")
    f.write(f"{'METEOR Score':<25} | {experiment3_meteor_val:.4f}\n")
    f.write(f"{'Epochs Completed':<25} | {len(losses)}\n")
    f.write(f"{'Predictions Evaluated':<25} | {len(pred_sentences)}\n")

print(f"Experiment 3 metrics table saved: {metrics_path}")


Comparison with Baseline and BiGRU

In [ ]:
baseline_dir = Path("results") / "baseline"
experiment1_dir = Path("results") / "Experiment_1"
experiment2_dir = Path("results") / "Experiment_2"
experiment3_dir = Path("results") / "Experiment_3"

comparison_path = experiment3_dir / "comparison_table.txt"
bigru_bilstm_loss_path = experiment3_dir / "experiment2_vs_experiment3_loss_side_by_side.png"
baseline_bilstm_loss_path = experiment3_dir / "baseline_vs_experiment3_loss_side_by_side.png"


def load_metrics_table(metrics_path):
    metrics = {}
    with open(metrics_path, "r", encoding="utf-8") as f:
        for line in f:
            if "|" in line and "Metric" not in line:
                name, value = line.split("|", 1)
                name = name.strip()
                value = value.strip()
                try:
                    metrics[name] = float(value)
                except ValueError:
                    metrics[name] = value
    return metrics


def load_training_losses(log_path):
    losses_from_log = []
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            if "Loss:" in line:
                losses_from_log.append(float(line.split("Loss:", 1)[1].split()[0]))
    return losses_from_log


def save_side_by_side_loss(left_losses, right_losses, left_title, right_title, output_path, suptitle):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    axes[0].plot(range(1, len(left_losses) + 1), left_losses, linewidth=2, color="steelblue")
    axes[0].set_title(left_title)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Training Loss")
    axes[0].grid(True)

    axes[1].plot(range(1, len(right_losses) + 1), right_losses, linewidth=2, color="darkorange")
    axes[1].set_title(right_title)
    axes[1].set_xlabel("Epoch")
    axes[1].grid(True)

    fig.suptitle(suptitle)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


experiment_dirs = {
    "Baseline": baseline_dir,
    "Experiment 1": experiment1_dir,
    "Experiment 2 BiGRU": experiment2_dir,
    "Experiment 3 BiLSTM": experiment3_dir,
}

all_metrics = {
    name: load_metrics_table(folder / "metrics.txt")
    for name, folder in experiment_dirs.items()
}

comparison_rows = [
    "Baseline vs Experiment 1 vs BiGRU vs BiLSTM",
    "=" * 104,
    f"{'Metric':<25} | {'Baseline':<12} | {'Experiment 1':<12} | {'BiGRU':<12} | {'BiLSTM':<12}",
    "-" * 104,
]

for metric in [
    "Final Training Loss",
    "BLEU Score",
    "BLEU-4 Score",
    "METEOR Score",
    "Epochs Completed",
    "Predictions Evaluated",
]:
    baseline_value = all_metrics["Baseline"].get(metric, 0.0)
    exp1_value = all_metrics["Experiment 1"].get(metric, 0.0)
    bigru_value = all_metrics["Experiment 2 BiGRU"].get(metric, 0.0)
    bilstm_value = all_metrics["Experiment 3 BiLSTM"].get(metric, 0.0)

    if all(isinstance(value, float) for value in [baseline_value, exp1_value, bigru_value, bilstm_value]):
        comparison_rows.append(
            f"{metric:<25} | {baseline_value:<12.4f} | {exp1_value:<12.4f} | {bigru_value:<12.4f} | {bilstm_value:<12.4f}"
        )
    else:
        comparison_rows.append(
            f"{metric:<25} | {baseline_value!s:<12} | {exp1_value!s:<12} | {bigru_value!s:<12} | {bilstm_value!s:<12}"
        )

comparison_text = "\n".join(comparison_rows)
print(comparison_text)

with open(comparison_path, "w", encoding="utf-8") as f:
    f.write(comparison_text + "\n")


baseline_losses = load_training_losses(baseline_dir / "training_log.txt")
experiment2_losses = load_training_losses(experiment2_dir / "training_log.txt")
experiment3_losses = load_training_losses(experiment3_dir / "training_log.txt")

save_side_by_side_loss(
    experiment2_losses,
    experiment3_losses,
    "Experiment 2 BiGRU",
    "Experiment 3 BiLSTM",
    bigru_bilstm_loss_path,
    "Experiment 2 vs Experiment 3",
)

save_side_by_side_loss(
    baseline_losses,
    experiment3_losses,
    "Baseline",
    "Experiment 3 BiLSTM",
    baseline_bilstm_loss_path,
    "Baseline vs Experiment 3",
)

print(f"Experiment 3 comparison table saved: {comparison_path}")
print(f"BiGRU vs BiLSTM side-by-side loss figure saved: {bigru_bilstm_loss_path}")
print(f"Baseline vs BiLSTM side-by-side loss figure saved: {baseline_bilstm_loss_path}")
